In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
data = pd.read_csv("cleaned_data.csv")
data.drop("Unnamed: 0",axis=1,inplace=True)
data.head(5)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,product_category_name,distance,delivery_days,estimated_days,ships_in,review_time,arrival_time,delivery_impression,estimated_del_impression,ship_impression
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,0a8556ac6be836b46b3e89920d59291c,delivered,2018-04-25,2018-04-25 22:15:09,2018-05-02 15:20:00,...,home_construction,845.34,14,27,7,5,OnTime,Fast,Slow,Fast
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,f2c7fc58a9de810828715166c672f10a,delivered,2018-06-26,2018-06-26 11:18:58,2018-06-28 14:18:00,...,auto,23.27,3,20,6,3,OnTime,Fast,Neutral,Fast
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,25b14b69de0b6e184ae6fe2755e478f9,delivered,2017-12-12,2017-12-14 09:52:34,2017-12-15 20:13:22,...,perfumery,27.31,6,23,14,3,OnTime,Fast,Neutral,Neutral
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,7a5d8efaaa1081f800628c30d2b0728f,delivered,2017-12-06,2017-12-06 12:13:20,2017-12-07 20:28:28,...,bed_bath_table,457.50,15,29,6,0,OnTime,Fast,Slow,Fast
4,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,7a5d8efaaa1081f800628c30d2b0728f,delivered,2017-12-06,2017-12-06 12:13:20,2017-12-07 20:28:28,...,bed_bath_table,457.50,15,29,6,1,OnTime,Fast,Slow,Fast


In [3]:
data.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'customer_id', 'order_status',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'review_score', 'review_creation_date',
       'review_answer_timestamp', 'customer_unique_id',
       'zip_code_prefix_customer', 'customer_city', 'customer_state',
       'geolocation_lat_customer', 'geolocation_lng_customer',
       'geolocation_city_customer', 'geolocation_state_customer', 'product_id',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'order_item_id', 'seller_id', 'shipping_limit_date', 'price',
       'freight_value', 'zip_code_prefix_seller', 'seller_city',
       'seller_state', 'geolocation_lat_seller', 'geolocation_lng_seller',
       'geolocation_city_selle

In [4]:
columns_to_drop = ['order_id','payment_sequential', 'customer_id','order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                   'order_delivered_customer_date', 'order_estimated_delivery_date', 'review_creation_date', 'review_answer_timestamp', 'customer_unique_id',
                   'zip_code_prefix_customer', 'geolocation_lat_customer', 'geolocation_lng_customer', 'geolocation_city_customer', 'geolocation_state_customer',
                   'product_id', 'zip_code_prefix_seller', 'geolocation_lat_seller', 'geolocation_lng_seller', 'geolocation_city_seller', 'geolocation_state_seller',
                   'delivery_days', 'estimated_days', 'ships_in', 'review_time', 'order_item_id']


data = data.drop(columns=columns_to_drop, axis=1)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112630 entries, 0 to 112629
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   payment_type                112630 non-null  object 
 1   payment_installments        112630 non-null  int64  
 2   payment_value               112630 non-null  float64
 3   review_score                112630 non-null  int64  
 4   customer_city               112630 non-null  object 
 5   customer_state              112630 non-null  object 
 6   product_description_lenght  112630 non-null  float64
 7   product_photos_qty          112630 non-null  float64
 8   product_weight_g            112630 non-null  float64
 9   product_length_cm           112630 non-null  float64
 10  product_height_cm           112630 non-null  float64
 11  product_width_cm            112630 non-null  float64
 12  seller_id                   112630 non-null  object 
 13  shipping_limit

In [6]:
data['review_score'].unique()

array([1, 5, 4, 2, 3], dtype=int64)

In [7]:
data['review_score'] = data['review_score'].map({
    1: 'very bad',
    2: 'bad',
    3: 'good',
    4: 'very good',
    5: 'excellent',
})
data["review_score"].unique()

array(['very bad', 'excellent', 'very good', 'bad', 'good'], dtype=object)

In [8]:
data['payment_installments'].unique()

array([ 8,  1,  2,  3,  6,  4, 10,  5,  7, 12,  9, 13, 15, 24, 11, 18, 14,
       20, 21, 17, 22,  0, 16, 23], dtype=int64)

In [9]:
def map_installments(value):
    if value >= 0 and value <= 8:
        return 'small installments'
    elif value >= 9 and value <= 16:
        return 'average installments'
    elif value >= 17 and value <= 24:
        return 'big installments'
    else:
        return 'Unknown'

data['payment_installments'] = data['payment_installments'].map(map_installments)
data['payment_installments'].unique()

array(['small installments', 'average installments', 'big installments'],
      dtype=object)

In [10]:
data['payment_value'].unique()

array([ 99.33,  24.39,  65.71, ..., 100.55, 238.16, 363.31])

In [11]:
def map_payment_value(value):
    if value >= 0 and value <= 2735:
        return 'very small payment value'
    elif value >= 2736 and value <= 5471:
        return 'small payment value'
    elif value >= 5472 and value <= 8207:
        return 'average payment value'
    elif value >= 8208 and value <= 10943:
        return 'big payment value'
    elif value >= 10944 and value <= 13665:
        return 'very big payment value'
    else:
        return 'Unknown'

data['payment_value'] = data['payment_value'].map(map_payment_value)
data['payment_value'].unique()

array(['very small payment value', 'small payment value',
       'average payment value', 'very big payment value'], dtype=object)

In [12]:
data['product_description_lenght'].unique()

array([ 921., 1274., 1536., ..., 2550., 2605., 2062.])

In [13]:
def map_product_description_lenght(value):
    if value >= 4 and value <= 1330:
        return 'small product description lenght'
    elif value >= 1331 and value <= 2661:
        return 'average product description lenght'
    elif value >= 2662 and value <= 3992:
        return 'large product description lenght'
    else:
        return 'Unknown'

data['product_description_lenght'] = data['product_description_lenght'].map(map_product_description_lenght)
data['product_description_lenght'].unique()

array(['small product description lenght',
       'average product description lenght',
       'large product description lenght'], dtype=object)

In [14]:
data['product_photos_qty'].unique()

array([ 8.,  2.,  1.,  3.,  4.,  6.,  7.,  5.,  9., 17., 10., 11., 13.,
       12., 18., 15., 20., 14., 19.])

In [15]:
def map_product_photos_qty(value):
    if value >= 1 and value <= 7:
        return 'small product photos qty'
    elif value >= 8 and value <= 14:
        return 'medium product photos qty'
    elif value >= 15 and value <= 20:
        return 'large product photos qty'
    else:
        return 'Unknown'

data['product_photos_qty'] = data['product_photos_qty'].map(map_product_photos_qty)
data['product_photos_qty'].unique()

array(['medium product photos qty', 'small product photos qty',
       'large product photos qty'], dtype=object)

In [16]:
data['product_weight_kg'] = data['product_weight_g']/1000 
data.drop(['product_weight_g'],axis=1,inplace=True)
data['product_weight_kg'].unique()

array([ 0.8  ,  0.15 ,  0.25 , ...,  4.517,  2.886, 10.875])

In [17]:
def map_product_weight(value):
    if value >= 0 and value <= 8:
        return 'very small product weight'
    elif value > 8 and value <= 16:
        return 'small product weight'
    elif value > 16 and value <= 24:
        return 'medium product weight'
    elif value > 24 and value <= 32:
        return 'large product weight'
    elif value > 32 and value <= 41:
        return 'very large product weight'
    else:
        return 'Unknown'

data['product_weight_kg'] = data['product_weight_kg'].map(map_product_weight)
data['product_weight_kg'].unique()

array(['very small product weight', 'medium product weight',
       'small product weight', 'large product weight',
       'very large product weight'], dtype=object)

In [18]:
data['size'] = (data['product_length_cm']*data['product_height_cm'])*data['product_width_cm']
data.drop(['product_length_cm','product_height_cm','product_width_cm'],axis=1,inplace=True)

In [19]:
data['size'].unique()

array([ 7803.,  1056.,  3360., ..., 36630., 57408., 69165.])

In [20]:
def map_size(value):
    if value >= 168 and value <= 59376:
        return 'very small size'
    elif value >= 59377 and value <= 118585:
        return 'small size'
    elif value >= 118586 and value <= 177794:
        return 'medium size'
    elif value >= 177795 and value <= 237003:
        return 'large size'
    elif value >= 237004 and value <= 296208:
        return 'very large size'
    else:
        return 'Unknown'

data['size'] = data['size'].map(map_size)
data['size'].unique()

array(['very small size', 'small size', 'medium size', 'large size',
       'very large size'], dtype=object)

In [21]:
data['price'].unique()

array([ 79.8 ,  17.  ,  56.99, ...,  17.7 , 384.93,  74.82])

In [22]:
def map_price(value):
    if value >= 0 and value <= 1347:
        return 'very small price'
    elif value >= 1347 and value <= 2695:
        return 'small price'
    elif value >= 2696 and value <= 4043:
        return 'average price'
    elif value >= 4044 and value <= 5391:
        return 'big price'
    elif value >= 5392 and value <= 6735:
        return 'very big price'
    else:
        return 'Unknown'

data['price'] = data['price'].map(map_price)
data['price'].unique()

array(['very small price', 'small price', 'average price', 'big price',
       'very big price'], dtype=object)

In [23]:
data['freight_value'].unique()

array([ 19.53,   7.39,   8.72, ..., 196.86, 113.65,  66.31])

In [24]:
def map_freight_value(value):
    if value >= 0 and value <= 137:
        return 'small freight value'
    elif value > 137 and value <= 275:
        return 'average freight value'
    elif value > 275 and value <= 410:
        return 'big freight value'
    else:
        return 'Unknown'

data['freight_value'] = data['freight_value'].map(map_freight_value)
data['freight_value'].unique()

array(['small freight value', 'average freight value',
       'big freight value'], dtype=object)

In [25]:
data['distance'].unique()

array([ 845.34,   23.27,   27.31, ..., 1446.2 ,  274.63,  388.92])

In [26]:
def map_freight_value(value):
    if value >= 0 and value <= 2904:
        return 'small distance'
    elif value > 2904 and value <= 5808:
        return 'average distance'
    elif value > 5808 and value <= 8711:
        return 'big distance'
    else:
        return 'Unknown'

data['distance'] = data['distance'].map(map_freight_value)
data['distance'].unique()

array(['small distance', 'average distance', 'big distance'], dtype=object)

In [27]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112630 entries, 0 to 112629
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype 
---  ------                      --------------   ----- 
 0   payment_type                112630 non-null  object
 1   payment_installments        112630 non-null  object
 2   payment_value               112630 non-null  object
 3   review_score                112630 non-null  object
 4   customer_city               112630 non-null  object
 5   customer_state              112630 non-null  object
 6   product_description_lenght  112630 non-null  object
 7   product_photos_qty          112630 non-null  object
 8   seller_id                   112630 non-null  object
 9   shipping_limit_date         112630 non-null  object
 10  price                       112630 non-null  object
 11  freight_value               112630 non-null  object
 12  seller_city                 112630 non-null  object
 13  seller_state                1

In [28]:
data_list = data.values.tolist()
print(data_list[:5])
te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.4, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

[['credit_card', 'small installments', 'very small payment value', 'very bad', 'teofilo otoni', 'MG', 'small product description lenght', 'medium product photos qty', '213b25e6f54661939f11710a6fddb871', '2018-05-02', 'very small price', 'small freight value', 'salto', 'SP', 'home_construction', 'small distance', 'OnTime', 'Fast', 'Slow', 'Fast', 'very small product weight', 'very small size'], ['credit_card', 'small installments', 'very small payment value', 'excellent', 'sao paulo', 'SP', 'small product description lenght', 'small product photos qty', 'eaf6d55068dea77334e8477d3878d89e', '2018-07-02', 'very small price', 'small freight value', 'sao paulo', 'SP', 'auto', 'small distance', 'OnTime', 'Fast', 'Neutral', 'Fast', 'very small product weight', 'very small size'], ['credit_card', 'small installments', 'very small payment value', 'excellent', 'sao paulo', 'SP', 'average product description lenght', 'small product photos qty', 'cc419e0650a3c5ba77189a1882b7556a', '2017-12-26', 've

In [29]:
sorted_by_support = frequent_itemsets.sort_values(by="support", ascending=False)
print(sorted_by_support.head(10).to_string(index=False))

 support                                                        itemsets
0.999050                                      (very small payment value)
0.998242                                                (small distance)
0.997763                                           (small freight value)
0.997292                      (very small payment value, small distance)
0.996937                 (very small payment value, small freight value)
0.996129                                              (very small price)
0.996022                           (small distance, small freight value)
0.995774                    (very small payment value, very small price)
0.995197 (very small payment value, small distance, small freight value)
0.994389                              (small distance, very small price)


In [30]:
sorted_by_confidence = rules.sort_values(by="confidence", ascending=False)
print(sorted_by_confidence.head(10).to_string(index=False))

                                                                                                                                                          antecedents                consequents  antecedent support  consequent support  support  confidence     lift  leverage  conviction  zhangs_metric
        (very small size, credit_card, OnTime, small product description lenght, very small price, small installments, small freight value, small product photos qty) (very small payment value)            0.506579             0.99905 0.506579         1.0 1.000951  0.000481         inf       0.001925
                                                            (SP, credit_card, very small size, very small price, Fast, small freight value, small product photos qty) (very small payment value)            0.532256             0.99905 0.532256         1.0 1.000951  0.000506         inf       0.002031
                                                          (SP, credit_card, very small size, OnTime,

In [31]:
sorted_by_lift = rules.sort_values(by="lift", ascending=False)
print(sorted_by_lift.head(10).to_string(index=False))

                                                                                  antecedents                                                                                                                                  consequents  antecedent support  consequent support  support  confidence     lift  leverage  conviction  zhangs_metric
(small distance, very small price, very small product weight, small freight value, excellent)                                      (very small payment value, very small size, OnTime, Fast, small installments, small product photos qty)            0.536669            0.758697 0.453263    0.844586 1.113206  0.046094    1.552649       0.219485
                     (excellent, very small product weight, small distance, very small price)                 (very small payment value, very small size, OnTime, Fast, small installments, small freight value, small product photos qty)            0.536722            0.758626 0.453263    0.844502 1.113200  0.046092  

In [32]:
sorted_by_leverage = rules.sort_values(by="leverage", ascending=False)
print(sorted_by_leverage.head(10).to_string(index=False))

                                                                                             antecedents                                                                                              consequents  antecedent support  consequent support  support  confidence     lift  leverage  conviction  zhangs_metric
                                     (Fast, OnTime, very small payment value, very small product weight)                      (small distance, very small size, very small price, small freight value, excellent)            0.802548            0.548211 0.486886    0.606675 1.106646  0.046921    1.148642       0.488062
                     (small distance, very small size, very small price, small freight value, excellent)                                      (Fast, OnTime, very small payment value, very small product weight)            0.548211            0.802548 0.486886    0.888137 1.106646  0.046921    1.765117       0.213305
                                          (excell

In [33]:
selected_columns = ['payment_type', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['credit_card', 'very bad'], ['credit_card', 'excellent'], ['credit_card', 'excellent'], ['credit_card', 'excellent'], ['credit_card', 'excellent']]
excellent -> credit_card
credit_card -> excellent


In [34]:
selected_columns = ['payment_value', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['very small payment value', 'very bad'], ['very small payment value', 'excellent'], ['very small payment value', 'excellent'], ['very small payment value', 'excellent'], ['very small payment value', 'excellent']]
excellent -> very small payment value
very small payment value -> excellent


In [35]:
selected_columns = ['customer_state', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['MG', 'very bad'], ['SP', 'excellent'], ['SP', 'excellent'], ['MG', 'excellent'], ['MG', 'excellent']]
SP -> excellent


In [36]:
selected_columns = ['seller_state', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['SP', 'very bad'], ['SP', 'excellent'], ['SP', 'excellent'], ['SP', 'excellent'], ['SP', 'excellent']]
excellent -> SP
SP -> excellent


In [37]:
selected_columns = ['arrival_time', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['OnTime', 'very bad'], ['OnTime', 'excellent'], ['OnTime', 'excellent'], ['OnTime', 'excellent'], ['OnTime', 'excellent']]
excellent -> OnTime
OnTime -> excellent


In [38]:
selected_columns = ['delivery_impression', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['Fast', 'very bad'], ['Fast', 'excellent'], ['Fast', 'excellent'], ['Fast', 'excellent'], ['Fast', 'excellent']]
Fast -> excellent
excellent -> Fast


In [39]:
selected_columns = ['distance', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['small distance', 'very bad'], ['small distance', 'excellent'], ['small distance', 'excellent'], ['small distance', 'excellent'], ['small distance', 'excellent']]
excellent -> small distance
small distance -> excellent


In [40]:
selected_columns = ['product_description_lenght', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['small product description lenght', 'very bad'], ['small product description lenght', 'excellent'], ['average product description lenght', 'excellent'], ['small product description lenght', 'excellent'], ['small product description lenght', 'excellent']]
excellent -> small product description lenght
small product description lenght -> excellent


In [41]:
selected_columns = ['price', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['very small price', 'very bad'], ['very small price', 'excellent'], ['very small price', 'excellent'], ['very small price', 'excellent'], ['very small price', 'excellent']]
excellent -> very small price
very small price -> excellent


In [42]:
selected_columns = ['freight_value', 'review_score']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['small freight value', 'very bad'], ['small freight value', 'excellent'], ['small freight value', 'excellent'], ['small freight value', 'excellent'], ['small freight value', 'excellent']]
excellent -> small freight value
small freight value -> excellent


In [43]:
selected_columns = ['arrival_time', 'delivery_impression']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['OnTime', 'Fast'], ['OnTime', 'Fast'], ['OnTime', 'Fast'], ['OnTime', 'Fast'], ['OnTime', 'Fast']]
Fast -> OnTime
OnTime -> Fast


In [44]:
selected_columns = ['distance', 'freight_value']
data_subset = data[selected_columns]

data_list = data_subset.values.tolist()
print(data_list[:5])

te = TransactionEncoder()
te_ary = te.fit(data_list).transform(data_list)
df = pd.DataFrame(te_ary, columns=te.columns_)
frequent_itemsets = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

formatted_rules = rules.apply(
    lambda row: f"{list(row['antecedents'])[0]} -> {list(row['consequents'])[0]}", axis=1)

for rule in formatted_rules:
    print(rule, end="\n")

[['small distance', 'small freight value'], ['small distance', 'small freight value'], ['small distance', 'small freight value'], ['small distance', 'small freight value'], ['small distance', 'small freight value']]
small distance -> small freight value
small freight value -> small distance
